# NB4 — Budget grid (RQ3)

v7 pipeline. Runs on Kaggle GPU.

## Setup required before running

1. From NB1, add the `xai-credit-preprocessed` Kaggle Dataset as input (test data).
2. From NB3, upload the `NB3-models` folder as a new Kaggle Dataset named `xai-credit-basins`,
   with the structure `xai-credit-basins/{dataset}/basin_01.pt ... basin_50.pt`, and add it as
   input to this notebook.

## What this notebook answers

RQ3: with a fixed inference budget (`M` basins x `T` MC-Dropout passes per basin = 50 forward
evaluations total), how does the *allocation* between M and T affect the quality of the
uncertainty-based gating signal? This does not require re-running SHAP/LIME — gating quality is
evaluated directly from predictions and uncertainty, so this notebook is comparatively cheap
despite covering many more configurations than NB3's four canonical models.

## Design

Six configurations, all with `M x T = 50`:

`(1,50), (2,25), (5,10), (10,5), (25,2), (50,1)`

The pool of 50 trained basins from NB3 is reused directly — no additional training. For any
config with `M < 50`, there are `C(50, M)` possible subsets of basins; `R_REPEATS` of them are
sampled at random (without replacement within each draw) and each is evaluated separately, which
gives a distribution of outcomes per configuration rather than a single point estimate. The
`M = 50` configuration has only one possible subset (all basins), so it is evaluated once.

Dropout is switched on whenever `T > 1` (an MC-Dropout pass makes sense only when there is more
than one pass to average); at `T = 1` it is switched off, so the single pass is the basin's own
best point estimate rather than one noisy draw — this matches how M4 was defined in NB3.

## Metrics computed per configuration

- `gating_auc_std`  — `AUC(1[y_pred != y_true], epistemic_unc)`, the standard selective-prediction
  reading of "gating": does uncertainty predict where the model is wrong.
- `gating_auc_classdiscrim` — `AUC(y_true, epistemic_unc)`, the reading used in the v6 version of
  this study (does uncertainty predict class membership). Kept for continuity, but renamed to
  make the distinction explicit — it answers a different question than the metric above.
- `aurc` — area under the risk-coverage curve (selective prediction, lower is better).
- `epi_pct_of_total` — mean epistemic uncertainty as a percentage of mean total uncertainty.
- `epi_cv` — coefficient of variation of epistemic uncertainty across samples.

## Outputs

`NB4-outputs/budget_grid_raw_{dataset}.csv`     — every repeat, every sample (for re-analysis)
`NB4-outputs/budget_grid_summary_{dataset}.csv` — mean + 95% CI per config per metric


## 1. Installs & imports

In [ ]:
!pip install fastparquet scikit-learn -q

import os
import json
import itertools
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score

import warnings
warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


## 2. Configuration

In [ ]:
class Config:
    DATA_DIR = "/kaggle/input/xai-credit-preprocessed"
    BASINS_DIR = "/kaggle/input/xai-credit-basins"
    OUTPUT_DIR = "/kaggle/working"
    SEED = 42
    DATASETS = ["home_credit", "taiwan", "gmsc"]

    N_BASINS_AVAILABLE = 50
    BUDGET_CONFIGS = [(1, 50), (2, 25), (5, 10), (10, 5), (25, 2), (50, 1)]
    R_REPEATS = 30            # random basin-subset draws per config (ignored when M = 50)

    HIDDEN_DIMS = [256, 128, 64]
    DROPOUT_RATE = 0.2

    EVAL_SPLIT = "test_natural"   # "test_balanced" or "test_natural"
    # v7 audit fix: was the balanced split. Gating/AURC/budget-grid metrics need the natural class
    # prevalence to define "misclassified" meaningfully -- a threshold on the balanced
    # 50:50 split does not reflect deployment. Consistent with the natural-split gating
    # table already used in NB6/NB7.

    if not os.path.exists(DATA_DIR):
        print(f"WARNING: {DATA_DIR} not found. Assuming local test run.")
        DATA_DIR = "../kaggle_outputs/xai-credit-preprocessed"
    if not os.path.exists(BASINS_DIR):
        print(f"WARNING: {BASINS_DIR} not found. Assuming local test run.")
        BASINS_DIR = "../kaggle_outputs/xai-credit-basins"


np.random.seed(Config.SEED)

## 3. Model + HB-MCD utilities (same definitions as NB3)

In [ ]:
class CreditMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims=(256, 128, 64), dropout_rate=0.2):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev_dim, h))
            layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            prev_dim = h
        layers.append(nn.Linear(prev_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


def enable_mc_dropout(model):
    model.eval()
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.train()
    return model


def load_basin(ds, basin_id, input_dim):
    model = CreditMLP(input_dim, Config.HIDDEN_DIMS, Config.DROPOUT_RATE).to(DEVICE)
    path = f"{Config.BASINS_DIR}/{ds}/basin_{basin_id:02d}.pt"
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    return model


def batched_mc_passes(model, X_t, T, dropout_on):
    if dropout_on:
        enable_mc_dropout(model)
    else:
        model.eval()
    out = np.zeros((T, X_t.shape[0]))
    with torch.no_grad():
        for t in range(T):
            logits = model(X_t)
            out[t] = torch.sigmoid(logits).cpu().numpy()
    return out


def binary_entropy(p, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)
    return -(p * np.log(p) + (1 - p) * np.log(1 - p))


def hb_mcd_decompose(preds_mtn):
    M, T, N = preds_mtn.shape
    p_bar_basin = preds_mtn.mean(axis=1)
    p_bar = p_bar_basin.mean(axis=0)

    aleatoric = binary_entropy(preds_mtn).mean(axis=(0, 1))
    intra = binary_entropy(p_bar_basin).mean(axis=0) - aleatoric
    inter = binary_entropy(p_bar) - binary_entropy(p_bar_basin).mean(axis=0)
    total = binary_entropy(p_bar)

    max_residual = np.max(np.abs(total - (aleatoric + intra + inter)))
    assert max_residual < 1e-6, f"HB-MCD decomposition mismatch: residual={max_residual}"

    return {
        "pred_mean": p_bar, "aleatoric_unc": aleatoric,
        "intra_unc": np.maximum(intra, 0.0), "inter_unc": np.maximum(inter, 0.0),
        "epistemic_unc": np.maximum(intra, 0.0) + np.maximum(inter, 0.0), "total_unc": total,
    }


def compute_aurc(uncertainty, errors):
    """Area under the risk-coverage curve. Samples sorted by ascending uncertainty (most
    confident first); risk = cumulative error rate at each coverage level. Lower AURC = better
    selective-prediction behavior (uncertainty correctly flags the model's mistakes)."""
    order = np.argsort(uncertainty)
    sorted_errors = errors[order]
    n = len(errors)
    cum_errors = np.cumsum(sorted_errors)
    coverage = np.arange(1, n + 1) / n
    risk = cum_errors / np.arange(1, n + 1)
    # np.trapz was removed in NumPy 2.2+ (renamed np.trapezoid) -- use a manual, version-
    # independent trapezoidal rule instead of depending on either name existing.
    return float(np.sum((risk[1:] + risk[:-1]) / 2.0 * np.diff(coverage)))


## 4. Evaluate a single (dataset, config, basin-subset) combination

In [ ]:
def evaluate_config(ds, X_t, y_true, basin_ids, T, input_dim):
    dropout_on = T > 1
    M = len(basin_ids)
    preds = np.zeros((M, T, len(y_true)))
    for i, bid in enumerate(basin_ids):
        model = load_basin(ds, bid, input_dim)
        preds[i] = batched_mc_passes(model, X_t, T, dropout_on)

    decomp = hb_mcd_decompose(preds)
    pred_mean = decomp["pred_mean"]
    epistemic = decomp["epistemic_unc"]
    total = decomp["total_unc"]

    # Base-rate threshold, not a hard 0.5 cutoff: the model is trained on the natural class
    # prevalence, so raw probabilities cluster near that prevalence and a 0.5 cutoff predicts
    # almost everyone as the majority class, collapsing "misclassified" onto "y_true == 1".
    # (v7 audit: this was the exact threshold bug already fixed in NB6/NB7 but missed here --
    # the symptom was gating_auc_std == gating_auc_classdiscrim to the last decimal.)
    tau = np.quantile(pred_mean, 1.0 - y_true.mean())
    y_pred = (pred_mean >= tau).astype(int)
    misclassified = (y_pred != y_true).astype(int)

    gating_auc_std = roc_auc_score(misclassified, epistemic) if misclassified.sum() > 0 else np.nan
    gating_auc_classdiscrim = roc_auc_score(y_true, epistemic)
    aurc = compute_aurc(epistemic, misclassified)
    epi_mean = epistemic.mean()
    epi_cv = float(epistemic.std() / max(epi_mean, 1e-12))
    epi_pct = float(epi_mean / max(total.mean(), 1e-12) * 100)

    return {
        "gating_auc_std": gating_auc_std, "gating_auc_classdiscrim": gating_auc_classdiscrim,
        "aurc": aurc, "epi_mean": float(epi_mean), "epi_cv": epi_cv, "epi_pct_of_total": epi_pct,
        "intra_mean": float(decomp["intra_unc"].mean()), "inter_mean": float(decomp["inter_unc"].mean()),
    }

## 5. Main loop: sweep the budget grid for each dataset

In [ ]:
def process_dataset(ds):
    print(f"\n{'=' * 70}")
    print(f"DATASET: {ds}  (budget grid, split={Config.EVAL_SPLIT})")
    print(f"{'=' * 70}")

    df = pd.read_parquet(f"{Config.DATA_DIR}/{ds}_{Config.EVAL_SPLIT}.parquet")
    X = df.drop(columns=["TARGET"]).values.astype(np.float32)
    y = df["TARGET"].values.astype(np.float32)
    input_dim = X.shape[1]
    X_t = torch.tensor(X, dtype=torch.float32).to(DEVICE)
    print(f"Eval set: {X.shape}, prevalence={y.mean():.4f}")

    rng = np.random.RandomState(Config.SEED)
    all_basins = np.arange(1, Config.N_BASINS_AVAILABLE + 1)

    raw_rows = []
    for config_id, (M, T) in enumerate(Config.BUDGET_CONFIGS):
        n_repeats = 1 if M == Config.N_BASINS_AVAILABLE else Config.R_REPEATS
        print(f"\n  Config M={M:2d} T={T:2d}  (M*T={M * T})  -- {n_repeats} repeat(s)")

        for rep in range(n_repeats):
            if M == Config.N_BASINS_AVAILABLE:
                basin_ids = list(all_basins)
            else:
                basin_ids = list(rng.choice(all_basins, size=M, replace=False))

            metrics = evaluate_config(ds, X_t, y, basin_ids, T, input_dim)
            metrics.update({"config_id": config_id, "M": M, "T": T, "repeat": rep,
                             "basin_ids": ",".join(str(b) for b in basin_ids)})
            raw_rows.append(metrics)

            if rep == 0 or rep == n_repeats - 1:
                print(f"    repeat {rep:2d} | gating_auc_std={metrics['gating_auc_std']:.4f} "
                      f"| aurc={metrics['aurc']:.4f} | epi_pct={metrics['epi_pct_of_total']:.3f}%")

    raw_df = pd.DataFrame(raw_rows)
    raw_path = f"{Config.OUTPUT_DIR}/budget_grid_raw_{ds}.csv"
    raw_df.to_csv(raw_path, index=False)
    print(f"\nRaw results saved -> {raw_path}  ({len(raw_df)} rows)")

    metric_cols = ["gating_auc_std", "gating_auc_classdiscrim", "aurc", "epi_mean", "epi_cv",
                   "epi_pct_of_total", "intra_mean", "inter_mean"]
    summary_rows = []
    for config_id, (M, T) in enumerate(Config.BUDGET_CONFIGS):
        sub = raw_df[raw_df["config_id"] == config_id]
        for metric in metric_cols:
            vals = sub[metric].dropna().values
            if len(vals) == 0:
                continue
            mean = float(np.mean(vals))
            if len(vals) > 1:
                ci_lo, ci_hi = np.percentile(vals, [2.5, 97.5])
            else:
                ci_lo, ci_hi = mean, mean
            summary_rows.append({
                "dataset": ds, "config_id": config_id, "M": M, "T": T, "metric": metric,
                "mean": mean, "ci_low": float(ci_lo), "ci_high": float(ci_hi), "n_repeats": len(vals),
            })

    summary_df = pd.DataFrame(summary_rows)
    summary_path = f"{Config.OUTPUT_DIR}/budget_grid_summary_{ds}.csv"
    summary_df.to_csv(summary_path, index=False)
    print(f"Summary saved -> {summary_path}")
    print(summary_df[summary_df["metric"] == "gating_auc_std"].to_string(index=False))


## 6. Run for all datasets

In [ ]:
for ds in Config.DATASETS:
    try:
        process_dataset(ds)
    except Exception as e:
        import traceback
        print(f"FAILED on {ds}: {e}")
        traceback.print_exc()

print("\nAll datasets processed. Download /kaggle/working as NB4-outputs.")
